# 13 — Xuất `sdc_iden` (+ head OS của 05b) sang TensorFlow Lite: 12 chuỗi → make/type/model

| | `sdc_iden.onnx` (09) | file TFLite của notebook này |
|---|---|---|
| input | `input` `string [N,12]` | `input` `string [N,12]` — cùng cột, cùng thứ tự (`FEATURES.md`) |
| output | `output` `string [N,3]` | `output` `string [N,3]` — `make`, `type`, `model` |
| head OS | không | `OS_RUN` đặt thì có: head OS của 05b + cascade **OS → type → make → model** |
| ngưỡng | ghi đè được qua input `thresholds` | **nung cố định** (TFLite không có input mặc định) |
| contract | `metadata_props["sdc.contract"]` | metadata `sdc.contract` trong flatbuffer |
| runtime | onnxruntime | LiteRT **+ Flex delegate** |

**`OS_RUN`** (mục 1) chọn run `Models/<ts>_os` của `05b_train_os_head`. Run đó chỉ đoán **OS**; make/type/model
vẫn do ba head của `sdc_iden` trả, rồi `cascade()` của 05b dùng OS để gạt câu mâu thuẫn (OS = Windows mà
type = Smartphone) và điền chỗ head iden bỏ trống (iOS → Apple / Smartphone). `OS_RUN = None` thì file chỉ
gồm `sdc_iden`, giống hệt ONNX.

**Vì sao cần Flex.** Đường 12 chuỗi cần tách token (`dhcp_prl`, TF-IDF), regex RE2 (luật hostname) và
ghép chuỗi (khoá vân tay L1) — TFLite builtin không có op nào trong số đó. Bật `SELECT_TF_OPS` thì các op
này chạy bằng kernel TensorFlow nhúng trong delegate; mọi thứ còn lại (rừng, hierarchy, ngưỡng, cascade, tra
nhãn) vẫn là op builtin. Runtime phía thiết bị phải link thêm Flex delegate — xem mục 6.

**Không đào lại bảng nào.** Rừng và encoder lấy từ bundle closed-set mà ONNX ghi trong `source_run` và từ
bundle OS; mọi bảng của `sdc_iden` (vân tay L1, OUI, mDNS `model=`, luật hostname theo thứ tự ưu tiên, mask
hierarchy, ngưỡng, nhãn) **trích thẳng từ graph ONNX**; luật OS và cascade lấy nguyên code của 05b. Mục 5 so
**từng nhãn** với chính các thứ đó chạy bằng Python.

Cần `tensorflow` (Python 3.14: `pip install --pre tensorflow`), `onnx`, `onnxruntime`, `pyarrow`.

In [1]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print("Configured for 88-core CPU")

Configured for 88-core CPU


In [2]:
# Nạp định nghĩa dùng chung từ 00_common.ipynb (không train).
from pathlib import Path

_cwd = Path.cwd().resolve()
_PIPELINE = next((d for p in (_cwd, *_cwd.parents)
                  for d in (p, p / "pipeline", p / "Code" / "pipeline")
                  if (d / "00_common.ipynb").is_file()), None)
assert _PIPELINE is not None, f"Không thấy pipeline/00_common.ipynb quanh {_cwd}"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_PIPELINE / "00_common.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import ast
import hashlib
import inspect
import itertools
import json
import re
import warnings
from datetime import datetime

import joblib
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import tensorflow as tf
from IPython.display import display
from tensorflow.lite.python import schema_py_generated as tfl_schema
from tensorflow.lite.tools import flatbuffer_utils

warnings.filterwarnings("ignore", message=".*tf.lite.Interpreter is deprecated.*")
ort.set_default_logger_severity(3)
print("TensorFlow", tf.__version__, "| onnxruntime", ort.__version__)

Configured for 88-core CPU
Gốc dự án: D:\01.AI_Security\02.SDC


Đã nạp hàm SDC từ 00_common.ipynb


TensorFlow 2.22.0-rc0 | onnxruntime 1.28.0


## 1. Chọn run, nạp định nghĩa dùng lại

Định nghĩa lấy theo tên (AST) từ notebook gốc thay vì chép lại: dựng input/cờ từ **09**
(`add_l0_columns`, `to_input_matrix`, `matrix_frame`, hằng contract), dựng rừng phẳng và sửa flatbuffer từ
**12** (`flatten_forest`, `rename_io`, `embed_metadata`), luật OS và cascade từ **05b** (`os_decide`,
`cascade`, `HOST_OS`...). Sửa ở notebook gốc là notebook này theo — và mục 5 bắt được nếu graph không theo kịp.

**`mac_is_random` không có trong 12 cột.** Head OS cần nó (luật VCI Android, chữ ký PRL, mDNS Apple, rừng,
cascade). Suy từ `mac_oui`: extractor (`mac_oui()` của 09) chỉ trả `<missing>` khi MAC ngẫu nhiên hoá hoặc hỏng,
mà router luôn biết MAC của thiết bị — nên `<missing>` được hiểu là **MAC ngẫu nhiên**. Caller gửi nguyên OUI
(09 cho phép) thì xét bit U/L `0x02`. Mục 5 kiểm quy ước này khớp `mac_is_random(MAC)` của 05b trên mọi dòng thật.

In [3]:
def import_defs(notebook, names, namespace):
    """Chạy các định nghĩa top-level (def/class/gán biến) có tên trong `names` của một notebook."""
    nb = json.loads(Path(notebook).read_text(encoding="utf-8"))
    wanted = []
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        try:
            tree = ast.parse("".join(cell["source"]))
        except SyntaxError:
            continue
        for node in tree.body:
            name = (node.name if isinstance(node, (ast.FunctionDef, ast.ClassDef)) else
                    node.targets[0].id if isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name)
                    else None)
            if name in names:
                wanted.append(node)
    exec(compile(ast.Module(body=wanted, type_ignores=[]), str(notebook), "exec"), namespace)
    missing = sorted(set(names) - set(namespace))
    assert not missing, f"{Path(notebook).name} thiếu: {missing}"
    return namespace


NB09 = import_defs(_PIPELINE / "out_model" / "09_build_sdc_iden.ipynb", [
    "MISSING", "UNKNOWN_LABEL", "ABSTAIN_LABEL", "KEY_SEP", "NOT_PRESENT", "HEADS", "MODEL_HEAD", "TIEBREAK",
    "ENCODER_STRINGS", "L0_COLUMNS", "INPUT_COLUMNS", "SOURCE_FLAGS", "SOURCE_EVIDENCE", "PRL_SEP",
    "FP_MODES", "OUT_LABELS", "IN_FEATURES", "_MAC_RE", "normalize_mac", "mac_is_random", "mac_oui",
    "_clean_text", "add_l0_columns", "to_input_matrix", "prl_options", "derive_numeric", "matrix_frame",
    "prepare_frame", "encode",
], {"re": re, "np": np, "pd": pd})
NB12 = import_defs(_PIPELINE / "out_model" / "12_export_tflite_os.ipynb", [
    "IO_NAMES", "float32_floor", "flatten_forest", "rename_io", "embed_metadata", "read_metadata",
    "CONTRACT_KEY",
], {"np": np, "flatbuffer_utils": flatbuffer_utils, "tfl_schema": tfl_schema})
HEADS, MODEL_HEAD, UNK = NB09["HEADS"], NB09["MODEL_HEAD"], NB09["UNKNOWN_LABEL"]
COLS = list(NB09["INPUT_COLUMNS"])
IO_NAMES = NB12["IO_NAMES"]
assert (NB09["IN_FEATURES"], NB09["OUT_LABELS"]) == IO_NAMES

IDEN_RUN = None                   # None = *_iden mới nhất
OS_RUN = "20261002_163153_os"     # run *_os của 05b; None = chỉ sdc_iden, không head OS

iden_dir = (MODELS / IDEN_RUN) if IDEN_RUN else sorted(p for p in MODELS.glob("*_iden")
                                                       if (p / "sdc_iden.onnx").is_file())[-1]
onnx_path = iden_dir / "sdc_iden.onnx"
onnx_model = onnx.load(str(onnx_path))
onnx_contract = json.loads({p.key: p.value for p in onnx_model.metadata_props}["sdc.contract"])
assert onnx_contract["format"] == "sdc-iden-onnx-v2", onnx_contract["format"]
assert onnx_contract["columns"] == COLS, "thứ tự cột của ONNX khác 09 hiện hành"

bundle = joblib.load(MODELS / onnx_contract["source_run"] / "model.joblib")
enc = bundle["encoder"]
LABELS = {h: [str(c) for c in bundle["models"][h].classes_] for h in HEADS}
assert LABELS == onnx_contract["labels"], "nhãn của bundle khác nhãn trong ONNX"
print(f"iden: {iden_dir.name}  |  rừng: {onnx_contract['source_run']}  |  "
      + ", ".join(f"{h} {len(LABELS[h])} lớp" for h in HEADS))

if OS_RUN:
    os_dir = MODELS / OS_RUN
    os_bundle = joblib.load(os_dir / "model.joblib")
    assert os_bundle["format"] == "sdc-os-v0", os_bundle["format"]
    NB05B = import_defs(_PIPELINE / "train_model" / "05b_train_os_head.ipynb", [
        "UNK", "OS_CLASSES", "OS_THRESHOLD", "NOT_PRESENT", "HOST_OS", "mac_is_random", "prl_set",
        "l0_os", "l1_os", "os_decide", "TYPE_OK", "GENERAL_PURPOSE_TYPES", "MAKE_OK", "cascade",
    ], {"re": re, "np": np, "MISSING": MISSING, "apply_encoder": apply_encoder})
    assert NB05B["UNK"] == UNK
    assert NB05B["OS_THRESHOLD"] == os_bundle["threshold"], "ngưỡng OS trong 05b lệch run đã lưu"
    assert list(NB05B["OS_CLASSES"]) == list(os_bundle["classes"])
    MIN_JACCARD = inspect.signature(NB05B["l1_os"]).parameters["min_jaccard"].default
    OS_LABELS = list(os_bundle["classes"]) + [UNK]
    print(f"OS  : {OS_RUN}  |  lớp {OS_LABELS}  |  ngưỡng rừng {os_bundle['threshold']}  |  "
          f"{len(os_bundle['signatures'])} chữ ký PRL (Jaccard ≥ {MIN_JACCARD})")
    out_dir, ARTIFACT = os_dir / "tflite_iden_os", "sdc_iden_os"
else:
    out_dir, ARTIFACT = iden_dir / "tflite", "sdc_iden"
print("đầu ra:", out_dir / f"{ARTIFACT}.tflite")

iden: 20261001_170502_iden  |  rừng: 20260917_105135_field_closedset  |  make 9 lớp, type 5 lớp, model 12 lớp


OS  : 20261002_163153_os  |  lớp ['iOS', 'Android', 'Windows', 'macOS', 'Linux', 'Embedded', '__unknown__']  |  ngưỡng rừng 0.6  |  20 chữ ký PRL (Jaccard ≥ 0.9)
đầu ra: D:\01.AI_Security\02.SDC\Models\20261002_163153_os\tflite_iden_os\sdc_iden_os.tflite


## 2. Trích bảng từ graph ONNX

Tên node do `GB` của 09 sinh nên có tiền tố cố định: `sdc/le/<mode>/<head>_*` (vân tay L1),
`sdc/le/oui_*`, `sdc/le/mdns_*`, `sdc/rx/<head>_*` + `sdc/host/<head>_*` (luật hostname), `sdc/mask/<head>_*`,
`sdc/lab/<head>_*`, `thresholds`. Luật hostname được 09 dựng **ngược** (luật đầu nằm ngoài cùng), nên đảo lại
thứ tự node là ra thứ tự ưu tiên. Kích thước mọi bảng phải khớp `sizes` trong contract ONNX.

In [4]:
def onnx_tables(model):
    attrs = lambda n: {a.name: onnx.helper.get_attribute_value(a) for a in n.attribute}
    inits = {i.name: onnx.numpy_helper.to_array(i) for i in model.graph.initializer}
    stem = lambda name, prefix: name[len(prefix):].rsplit("_", 1)[0]
    l1 = {mode: {} for mode, _, _ in NB09["FP_MODES"]}
    oui, mdns, rx, host = {}, {}, {}, {h: [] for h in HEADS}
    for node in model.graph.node:
        out = node.output[0]
        if node.op_type == "LabelEncoder" and out.startswith("sdc/le/"):
            a = attrs(node)
            assert a["default_int64"] == -1
            keys = [k.decode() for k in a["keys_strings"]]
            book = {} if keys == ["\x00__empty__"] else dict(zip(keys, map(int, a["values_int64s"])))
            name = stem(out, "sdc/le/")
            if name == "oui":
                oui = book
            elif name == "mdns":
                mdns = book
            else:
                mode, head = name.split("/")
                l1[mode][head] = book
        elif node.op_type == "RegexFullMatch" and out.startswith("sdc/rx/"):
            rx[out] = (stem(out, "sdc/rx/"), attrs(node)["pattern"].decode())
        elif node.op_type == "Where" and out.startswith("sdc/host/"):
            head, pattern = rx[node.input[0]]
            host[head].append((pattern, int(inits[node.input[1]].ravel()[0])))
    find = lambda prefix: next(v for k, v in inits.items() if k.startswith(prefix))
    return {
        "l1": l1, "oui": oui, "mdns_model": mdns,
        "hostname": {h: list(reversed(rules)) for h, rules in host.items()},
        "masks": {h: find(f"sdc/mask/{h}_").astype(np.float32) for h in ("make", "type")},
        "labels": {h: [s.decode() if isinstance(s, bytes) else str(s) for s in find(f"sdc/lab/{h}_").ravel()]
                   for h in HEADS},
        "thresholds": inits["thresholds"].astype(np.float32),
    }


T = onnx_tables(onnx_model)
sizes = onnx_contract["sizes"]
assert {f"{m}/{h}": len(b) for m, hs in T["l1"].items() for h, b in hs.items()} == sizes["l1"]
assert (len(T["oui"]), len(T["mdns_model"])) == (sizes["oui"], sizes["mdns_model"])
assert sum(map(len, T["hostname"].values())) == sizes["hostname"]
assert T["labels"] == LABELS
assert np.allclose(T["thresholds"], [onnx_contract["thresholds"][h] for h in HEADS])
for h in ("make", "type"):
    assert T["masks"][h].shape == (len(LABELS[MODEL_HEAD]), len(LABELS[h]))
print("L1:", sizes["l1"])
print(f"OUI {len(T['oui'])}, mDNS model= {len(T['mdns_model'])}, hostname "
      + ", ".join(f"{h} {len(r)}" for h, r in T["hostname"].items()))

L1: {'fp_full/make': 2, 'fp_full/type': 2, 'fp_full/model': 2, 'fp_dhcp/make': 5, 'fp_dhcp/type': 5, 'fp_dhcp/model': 5, 'fp_tls/make': 3, 'fp_tls/type': 3, 'fp_tls/model': 3}
OUI 6, mDNS model= 9, hostname make 7, type 5, model 5


## 3. Graph TF

**Head iden** — đi đúng thứ tự `build_model` / `reference()` của 09:

1. **Cột và cờ** — tách 12 cột; `has_*` theo `SOURCE_EVIDENCE`; `dhcp_opt_k` = có token `k` trong
   `dhcp_prl` tách theo `,`; `dhcp_prl_len` = số token chữ số **không trùng token đứng trước** (tam giác
   dưới của ma trận so cặp, như `Trilu` của 09).
2. **Encoder** — ordinal: so chuỗi với danh sách category, trượt → `-1`; TF-IDF: lowercase, tách token đúng
   `token_pattern`, đếm theo vocab, nhân idf, chuẩn hoá L2 trong từng khối.
3. **Rừng** — 3 head, duyệt cây bằng gather như notebook 12.
4. **Gộp** — L0 (mDNS > OUI > hostname) và L1 (`fp_full` → `fp_dhcp` → `fp_tls`, `-2` dừng chuỗi), head
   `model` chạy trước và lái mask hierarchy cho `make`/`type`, rồi ngưỡng theo số nguồn → `__unknown__`.

**Head OS** (khi có `OS_RUN`) — đi đúng `os_decide()` của 05b:

5. **L0** — `HOST_OS` trên hostname đã `strip().lower()` (`re.search` → full-match `.*(…).*`; `\b` của Python
   là Unicode nên đổi thành `[^\p{L}\p{N}_]|$`), rồi VCI `msft ` / `android-dhcp-` (chỉ khi MAC ngẫu nhiên).
6. **L1** — tập option của `dhcp_prl` (mọi dãy chữ số, bỏ số 0 đầu như `int()`), Jaccard với từng chữ ký,
   tập OS của các chữ ký đạt max; trả lời khi ≥ `MIN_JACCARD` và đúng một OS (Embedded + MAC ngẫu nhiên thì bỏ
   qua). Rồi mDNS `_companion link` / `_rdlink` + MAC ngẫu nhiên → iOS. Tỉ số Jaccard là phân số nhỏ: hai
   phân số bằng nhau cho đúng cùng một float32, nên so bằng và so max trên float32 là chính xác.
7. **L2** — rừng OS; argmax lấy chỉ số đầu khi hoà (như numpy), trả lời khi conf ≥ ngưỡng, trừ khi lớp khác
   Embedded mà cửa sổ không có DHCP lẫn mDNS.
8. **Cascade** — `cascade()` của 05b trên chỉ số nhãn: gạt type/make mâu thuẫn OS, điền type/make từ OS,
   make hoặc type `__unknown__` thì model cũng `__unknown__`.

**Nhãn** — một bảng nhãn chung cho 3 head, gather bằng chỉ số `[N,3]` (op builtin; `PACK` chuỗi thì không).
Bảng tra nhỏ (≤ 61 khoá) nên tra bằng so bằng chuỗi với cả danh sách — không cần hashtable.

In [5]:
VCI_PATTERN, SPACE_PATTERN = r"[^:\-_. ]+", r"[^ ]+"


def encoder_spec(encoder, n_features):
    """Mọi thứ graph cần để dựng lại đúng vector của `ColumnTransformer`."""
    ct = encoder["ct"]
    cats = {c: [str(v) for v in vals]
            for c, vals in zip(encoder["cat_cols"], ct.named_transformers_["cat"].categories_)}
    text = {}
    for c in encoder["text_cols"]:
        vec = ct.named_transformers_[f"t_{c}"]
        p = vec.get_params()
        assert (p["analyzer"], p["ngram_range"], p["norm"], p["sublinear_tf"], p["use_idf"]) == ("word", (1, 1), "l2", False, True)
        assert p["lowercase"] and p["preprocessor"] is None and p["tokenizer"] is None and p["stop_words"] is None
        assert vec.token_pattern in (VCI_PATTERN, SPACE_PATTERN), vec.token_pattern
        text[c] = {"vocab": [str(t) for t in vec.get_feature_names_out()],
                   "idf": vec.idf_.astype(np.float32), "split_punct": vec.token_pattern == VCI_PATTERN}
    spec = {"num": list(encoder["num_cols"]), "cat": cats, "text": text, "width": int(n_features)}
    assert len(spec["num"]) + len(cats) + sum(len(t["vocab"]) for t in text.values()) == n_features
    assert set(cats) | set(text) <= set(COLS)
    return spec


SPEC = {"iden": encoder_spec(enc, bundle["models"]["make"].n_features_in_)}
FLAT = {h: NB12["flatten_forest"](bundle["models"][h]) for h in HEADS}
assert set(SPEC["iden"]["cat"]) | set(SPEC["iden"]["text"]) == set(NB09["ENCODER_STRINGS"])
if OS_RUN:
    SPEC["os"] = encoder_spec(os_bundle["encoder"], os_bundle["model"].n_features_in_)
    FLAT["os"] = NB12["flatten_forest"](os_bundle["model"])
    assert set(SPEC["os"]["num"]) <= set(SPEC["iden"]["num"]) | {"mac_is_random"}
for name, s in SPEC.items():
    print(f"encoder {name}: {s['width']} feature = num {len(s['num'])} + cat {len(s['cat'])} + "
          + " + ".join(f"{c} {len(t['vocab'])}" for c, t in s["text"].items()))
print("rừng:", {h: (len(f["roots"]), len(f["feature"]), f["depth"]) for h, f in FLAT.items()})

# --- Hằng của head OS + cascade -------------------------------------------------------------
OUI_RE = re.compile(r"[0-9a-fA-F]{2}:[0-9a-fA-F]{2}:[0-9a-fA-F]{2}")


def mac_random_from_oui(value):
    """Quy ước mac_is_random từ cột `mac_oui` — bản Python của `SdcModule.mac_random`."""
    v = str(value)
    if v in NB09["NOT_PRESENT"]:
        return 1                              # extractor chỉ trả <missing> khi MAC ngẫu nhiên hoá
    return int(bool(OUI_RE.fullmatch(v)) and int(v[:2], 16) & 0x02 != 0)


def search_as_fullmatch(pattern):
    """`re.search` (Python) -> full-match RE2. `\b` cuối mẫu là biên phải Unicode."""
    if r"\b" in pattern:
        assert pattern.endswith(r"\b") and pattern.count(r"\b") == 1, pattern
        pattern = pattern[:-2] + r"(?:[^\p{L}\p{N}_]|$)"
    return f"(?s:.*)(?:{pattern})(?s:.*)"


# Khoảng trắng mà str.strip() của Python bỏ: \t\n\v\f\r, \x1c-\x1f, \x85 và mọi ký tự nhóm Z.
PY_SPACE = r"[\t\n\x{0b}\f\r\x{1c}-\x{1f}\x{85}\p{Z}]"

if OS_RUN:
    OSI = {c: i for i, c in enumerate(OS_LABELS)}
    HOST_OS_RE = [(search_as_fullmatch(p), OSI[o]) for p, o in NB05B["HOST_OS"]]
    FOREST_OS = np.int32([OSI[str(c)] for c in os_bundle["model"].classes_])
    SIG_KEYS = list(os_bundle["signatures"])
    SIG_OPTIONS = sorted({o for k in SIG_KEYS for o in k})
    SIG_MEMBER = np.float32([[o in k for k in SIG_KEYS] for o in SIG_OPTIONS])                   # [U, K]
    SIG_SIZE = np.float32([len(k) for k in SIG_KEYS])
    SIG_OWNER = np.float32([[c in os_bundle["signatures"][k] for c in OS_LABELS] for k in SIG_KEYS])  # [K, O]
    TYPE_LABELS, MAKE_LABELS = LABELS["type"], LABELS["make"]
    CAS = {
        "type_rule": np.bool_([o in NB05B["TYPE_OK"] for o in OS_LABELS]),
        "type_ok": np.float32([[t in NB05B["TYPE_OK"].get(o, ()) for t in TYPE_LABELS] for o in OS_LABELS]),
        "general": np.bool_([t in NB05B["GENERAL_PURPOSE_TYPES"] for t in TYPE_LABELS]),
        "make_rule": np.bool_([o in NB05B["MAKE_OK"] for o in OS_LABELS]),
        "make_ok": np.float32([[m in NB05B["MAKE_OK"].get(o, ()) for m in MAKE_LABELS] for o in OS_LABELS]),
    }


class SdcModule(tf.Module):
    def __init__(self):
        super().__init__()
        spec = [tf.TensorSpec([None, len(COLS)], tf.string, name=IO_NAMES[0])]
        self.run = tf.function(self._run, input_signature=spec)
        self.debug = tf.function(self._core, input_signature=spec)
        if OS_RUN:
            self.cascade_fn = tf.function(self.cascade, input_signature=[
                tf.TensorSpec([None], tf.int32), tf.TensorSpec([None], tf.bool)]
                + [tf.TensorSpec([None], tf.int32)] * 3)

    # --- tiện ích ----------------------------------------------------------------------
    @staticmethod
    def present(x):
        bad = tf.equal(tf.expand_dims(x, 1), tf.constant(list(NB09["NOT_PRESENT"])))
        return tf.logical_not(tf.reduce_any(bad, axis=1))

    @staticmethod
    def lookup(x, book):
        """Chuỗi -> giá trị trong `book`, trượt -> -1 (như LabelEncoder default)."""
        if not book:
            return tf.fill(tf.shape(x), -1)
        eq = tf.equal(tf.expand_dims(x, 1), tf.constant(list(book)))
        pos = tf.argmax(tf.cast(eq, tf.int32), axis=1, output_type=tf.int32)
        return tf.where(tf.reduce_any(eq, axis=1),
                        tf.gather(tf.constant(list(book.values()), tf.int32), pos), -1)

    @staticmethod
    def tokens(x, sep):
        return tf.strings.split(x, sep=sep).to_tensor(default_value="")      # [N, M]

    @staticmethod
    def first_occurrence(tok):
        """[N, M] bool: token này không trùng token nào đứng trước nó."""
        same = tf.cast(tf.equal(tf.expand_dims(tok, 2), tf.expand_dims(tok, 1)), tf.float32)
        earlier = tf.linalg.band_part(same, -1, 0) - tf.linalg.band_part(same, 0, 0)   # j < i
        return tf.equal(tf.reduce_sum(earlier, axis=2), 0.0)

    @staticmethod
    def py_strip_lower(x):
        """`str(x).strip().lower()` của Python."""
        x = tf.strings.regex_replace(x, f"^{PY_SPACE}+|{PY_SPACE}+$", "")
        return tf.strings.lower(x, encoding="utf-8")

    def columns(self, inp):
        return {c: inp[:, i] for i, c in enumerate(COLS)}

    def mac_random(self, oui):
        bit = tf.strings.regex_full_match(oui, "[0-9a-fA-F][2367abefABEF]:[0-9a-fA-F]{2}:[0-9a-fA-F]{2}")
        return tf.logical_or(tf.logical_not(self.present(oui)), bit)

    # --- cờ + encoder -------------------------------------------------------------------
    def derived(self, col):
        out = {}
        for flag, sources in NB09["SOURCE_EVIDENCE"].items():
            hit = self.present(col[sources[0]])
            for c in sources[1:]:
                hit = tf.logical_or(hit, self.present(col[c]))
            out[flag] = tf.cast(hit, tf.float32)
        tok = self.tokens(col["dhcp_prl"], NB09["PRL_SEP"])
        for c in SPEC["iden"]["num"]:
            if c.startswith("dhcp_opt_"):
                code = c.rsplit("_", 1)[1]
                out[c] = tf.cast(tf.reduce_any(tf.equal(tok, code), axis=1), tf.float32)
        digit = tf.strings.regex_full_match(tok, "[0-9]+")
        out["dhcp_prl_len"] = tf.reduce_sum(
            tf.cast(tf.logical_and(digit, self.first_occurrence(tok)), tf.float32), axis=1)
        out["mac_is_random"] = tf.cast(self.mac_random(col["mac_oui"]), tf.float32)
        return out

    def tfidf(self, x, spec):
        x = tf.strings.lower(x, encoding="utf-8")
        if spec["split_punct"]:
            x = tf.strings.regex_replace(x, r"[:\-_.]", " ")
        tok = self.tokens(x, " ")
        counts = tf.reduce_sum(tf.cast(tf.equal(tf.expand_dims(tok, 2), tf.constant(spec["vocab"])),
                                       tf.float32), axis=1)                         # [N, V]
        v = counts * tf.constant(spec["idf"])
        norm = tf.sqrt(tf.reduce_sum(v * v, axis=1, keepdims=True))
        return tf.math.divide_no_nan(v, norm)

    def encode(self, col, derived, spec):
        parts = [tf.stack([derived[c] for c in spec["num"]], axis=1)]
        parts.append(tf.stack([tf.cast(self.lookup(col[c], dict(zip(cats, range(len(cats))))), tf.float32)
                               for c, cats in spec["cat"].items()], axis=1))
        parts += [self.tfidf(col[c], t) for c, t in spec["text"].items()]
        return tf.concat(parts, axis=1)

    @staticmethod
    def forest(x, f, width):
        n = tf.shape(x)[0]
        flat_x = tf.reshape(x, [-1])
        row = tf.expand_dims(tf.range(n) * width, 1)
        node = tf.tile(tf.expand_dims(tf.constant(f["roots"]), 0), tf.stack([n, 1]))
        feature, threshold = tf.constant(f["feature"]), tf.constant(f["threshold"])
        left, right = tf.constant(f["left"]), tf.constant(f["right"])
        for _ in range(int(f["depth"])):
            xv = tf.gather(flat_x, tf.gather(feature, node) + row)
            node = tf.where(xv <= tf.gather(threshold, node), tf.gather(left, node), tf.gather(right, node))
        return tf.reduce_mean(tf.gather(tf.constant(f["value"]), node), axis=1)

    # --- head iden: L0/L1 + rừng + hierarchy + ngưỡng -------------------------------------
    def l1(self, col, flags, head):
        answer, decided = None, None
        for mode, cols, mode_flags in NB09["FP_MODES"]:
            parts = []
            for c in cols:
                parts += [col[c], tf.fill(tf.shape(col[c]), NB09["KEY_SEP"])]
            key = tf.strings.join(parts[:-1]) if len(cols) > 1 else col[cols[0]]
            raw = self.lookup(key, T["l1"][mode].get(head, {}))
            usable = tf.ones_like(raw, dtype=tf.bool)
            for flag in mode_flags:
                usable = tf.logical_and(usable, tf.equal(flags[flag], 1.0))
            for c in cols:
                usable = tf.logical_and(usable, self.present(col[c]))
            raw = tf.where(usable, raw, -1)
            fresh = tf.where(raw >= 0, raw, -1)
            if answer is None:
                answer, decided = fresh, tf.not_equal(raw, -1)
            else:
                answer = tf.where(decided, answer, fresh)
                decided = tf.logical_or(decided, tf.not_equal(raw, -1))
        return answer

    def l0(self, col, head):
        out = []
        if head == MODEL_HEAD and T["mdns_model"]:
            out.append(self.lookup(col["mdns_model"], T["mdns_model"]))
        if head == "make" and T["oui"]:
            out.append(self.lookup(col["mac_oui"], T["oui"]))
        host = None
        for pattern, idx in reversed(T["hostname"][head]):
            match = tf.strings.regex_full_match(col["dhcp_hostname"], pattern)
            host = tf.where(match, idx, host if host is not None else tf.fill(tf.shape(match), -1))
        if host is not None:
            out.append(host)
        return out

    def fuse(self, head, p, col, flags, n_sources, allow=None):
        n_class = len(LABELS[head])

        def gate(v):
            if allow is None:
                return v
            ok = tf.reduce_sum(allow * tf.one_hot(tf.maximum(v, 0), n_class), axis=1) > 0
            return tf.where(ok, v, -1)

        gated = [gate(c) for c in self.l0(col, head)]
        override = gate(self.l1(col, flags, head))
        for c in reversed(gated):
            override = tf.where(c >= 0, c, override)
        has_override = override >= 0
        tie = tf.constant(np.arange(n_class, dtype=np.float32) * np.float32(NB09["TIEBREAK"]))
        forest_idx = tf.argmax(p - tie, axis=1, output_type=tf.int32)
        conf = tf.where(has_override, 1.0, tf.reduce_max(p, axis=1))
        index = tf.where(has_override, override, forest_idx)
        thr = tf.gather(tf.constant(T["thresholds"][HEADS.index(head)]), n_sources)
        return tf.where(conf < thr, LABELS[head].index(NB09["ABSTAIN_LABEL"]), index)

    # --- head OS: os_decide() của 05b -----------------------------------------------------
    def os_head(self, col, flags, x_os, random):
        unk = OSI[UNK]
        h = self.py_strip_lower(col["dhcp_hostname"])
        h_ok = tf.logical_not(tf.reduce_any(
            tf.equal(tf.expand_dims(h, 1), tf.constant(["", MISSING, "nan", "none"])), axis=1))
        host = tf.fill(tf.shape(h), -1)
        for pattern, idx in reversed(HOST_OS_RE):            # luật đầu tiên khớp thắng
            host = tf.where(tf.strings.regex_full_match(h, pattern), idx, host)
        host = tf.where(h_ok, host, -1)

        v = self.py_strip_lower(col["dhcp_vci"])
        android = tf.logical_and(tf.strings.regex_full_match(v, "(?s)android-dhcp-.*"), random)
        vci = tf.where(tf.strings.regex_full_match(v, "(?s)msft .*"), OSI["Windows"],
                       tf.where(android, OSI["Android"], -1))

        # L1 PRL — tập option = mọi dãy chữ số, bỏ số 0 đầu (như int()), không tính trùng
        tok = self.tokens(tf.strings.regex_replace(col["dhcp_prl"], "[^0-9]+", " "), " ")
        tok = tf.strings.regex_replace(tok, "^0+([0-9])", r"\1")
        keep = tf.logical_and(tf.not_equal(tok, ""), self.first_occurrence(tok))
        n_opt = tf.reduce_sum(tf.cast(keep, tf.float32), axis=1)                          # |tập option|
        has = tf.reduce_any(tf.logical_and(
            tf.equal(tf.expand_dims(tok, 2), tf.constant([str(o) for o in SIG_OPTIONS])),
            tf.expand_dims(keep, 2)), axis=1)                                             # [N, U]
        inter = tf.matmul(tf.cast(has, tf.float32), tf.constant(SIG_MEMBER))             # [N, K]
        ratio = inter / (tf.expand_dims(n_opt, 1) + tf.constant(SIG_SIZE) - inter)
        best = tf.reduce_max(ratio, axis=1)
        tied = tf.cast(tf.equal(ratio, tf.expand_dims(best, 1)), tf.float32)
        owners = tf.reduce_max(tf.expand_dims(tied, 2) * tf.constant(SIG_OWNER), axis=1)  # [N, O]
        prl_os = tf.argmax(owners, axis=1, output_type=tf.int32)
        prl_ok = (n_opt > 0) & (best >= MIN_JACCARD) & tf.equal(tf.reduce_sum(owners, axis=1), 1.0)
        prl_ok = prl_ok & tf.logical_not(tf.equal(prl_os, OSI["Embedded"]) & random)

        mdns = random & tf.strings.regex_full_match(col["mdns_tokens"], "(?s).*(?:_companion link|_rdlink).*")

        # L2 rừng — argmax lấy chỉ số đầu khi hoà, như numpy
        p = self.forest(x_os, FLAT["os"], SPEC["os"]["width"])
        pmax = tf.reduce_max(p, axis=1)
        n_class = len(FOREST_OS)
        k = tf.reduce_min(tf.where(tf.equal(p, tf.expand_dims(pmax, 1)), tf.range(n_class), n_class), axis=1)
        forest_os = tf.gather(tf.constant(FOREST_OS), k)
        tls_only = tf.logical_not(tf.equal(flags["has_dhcp"], 1.0) | tf.equal(flags["has_mdns"], 1.0))
        forest_ok = (pmax >= float(os_bundle["threshold"])) & tf.logical_not(
            tf.not_equal(forest_os, OSI["Embedded"]) & tls_only)

        os_idx = tf.where(forest_ok, forest_os, unk)
        os_idx = tf.where(mdns, OSI["iOS"], os_idx)
        os_idx = tf.where(prl_ok, prl_os, os_idx)
        os_idx = tf.where(vci >= 0, vci, os_idx)
        return tf.where(host >= 0, host, os_idx)

    def cascade(self, os_idx, random, make, type_, model):
        """`cascade()` của 05b trên chỉ số nhãn."""
        t_unk, m_unk, mo_unk = (LABELS[h].index(UNK) for h in ("type", "make", MODEL_HEAD))
        known = tf.not_equal(os_idx, OSI[UNK])
        is_os = lambda *names: tf.reduce_any(tf.stack([tf.equal(os_idx, OSI[n]) for n in names]), axis=0)
        pick = lambda table, idx, n: tf.reduce_sum(tf.gather(tf.constant(table), os_idx) * tf.one_hot(idx, n), axis=1) > 0

        emb_general = is_os("Embedded") & tf.gather(tf.constant(CAS["general"]), type_)
        conflict = (tf.not_equal(type_, t_unk) & tf.gather(tf.constant(CAS["type_rule"]), os_idx)
                    & tf.logical_not(pick(CAS["type_ok"], type_, len(TYPE_LABELS))))
        type_ = tf.where(known & (emb_general | conflict), t_unk, type_)
        empty = known & tf.equal(type_, t_unk)
        phone = empty & (is_os("iOS") | (is_os("Android") & random))
        laptop = empty & tf.logical_not(phone) & is_os("Windows", "macOS")
        type_ = tf.where(phone, TYPE_LABELS.index("Smartphone"), tf.where(laptop, TYPE_LABELS.index("Laptop"), type_))

        make_rule = tf.gather(tf.constant(CAS["make_rule"]), os_idx)
        bad = tf.not_equal(make, m_unk) & make_rule & tf.logical_not(pick(CAS["make_ok"], make, len(MAKE_LABELS)))
        make = tf.where(known & bad, m_unk, make)
        apple = MAKE_LABELS.index("Apple")
        make = tf.where(known & tf.equal(make, apple) & tf.logical_not(is_os("iOS", "macOS")), m_unk, make)
        make = tf.where(known & tf.equal(make, m_unk) & make_rule, apple, make)

        drop = (tf.equal(make, m_unk) | tf.equal(type_, t_unk)) & tf.not_equal(model, mo_unk)
        return make, type_, tf.where(drop, mo_unk, model)

    # --- lắp ráp ------------------------------------------------------------------------
    def _core(self, input):
        col = self.columns(input)
        flags = self.derived(col)
        x = self.encode(col, flags, SPEC["iden"])
        n_sources = tf.cast(tf.add_n([flags[f] for f in NB09["SOURCE_FLAGS"]]), tf.int32)
        proba = {h: self.forest(x, FLAT[h], SPEC["iden"]["width"]) for h in HEADS}
        idx = {MODEL_HEAD: self.fuse(MODEL_HEAD, proba[MODEL_HEAD], col, flags, n_sources)}
        for h in ("make", "type"):
            row = tf.gather(tf.constant(T["masks"][h]), idx[MODEL_HEAD])
            idx[h] = self.fuse(h, proba[h] * row, col, flags, n_sources, allow=row)
        out = {"x_iden": x, "iden": tf.stack([idx[h] for h in HEADS], axis=1)}
        if OS_RUN:
            random = tf.equal(flags["mac_is_random"], 1.0)
            x_os = self.encode(col, flags, SPEC["os"])
            os_idx = self.os_head(col, flags, x_os, random)
            idx["make"], idx["type"], idx[MODEL_HEAD] = self.cascade(
                os_idx, random, idx["make"], idx["type"], idx[MODEL_HEAD])
            out.update(x_os=x_os, os=os_idx)
        out["final"] = tf.stack([idx[h] for h in HEADS], axis=1)
        return out

    def _run(self, input):
        offset = np.cumsum([0] + [len(LABELS[h]) for h in HEADS])[:-1]
        table = tf.constant([lab for h in HEADS for lab in LABELS[h]])
        return {IO_NAMES[1]: tf.gather(table, self._core(input)["final"] + tf.constant(offset, tf.int32))}


module = SdcModule()

encoder iden: 1088 feature = num 31 + cat 5 + dhcp_vci 32 + dns_tokens 575 + mdns_tokens 118 + tls_sni_tokens 327
encoder os: 187 feature = num 32 + cat 5 + dhcp_vci 32 + mdns_tokens 118
rừng: {'make': (250, 16570, 31), 'type': (250, 13196, 24), 'model': (250, 19486, 30), 'os': (300, 13052, 20)}


## 4. Convert

`TFLITE_BUILTINS` trước, `SELECT_TF_OPS` cho phần còn lại. Danh sách op Flex in ra dưới đây chính là thứ
runtime trên thiết bị phải có.

In [6]:
converter = tf.lite.TFLiteConverter.from_concrete_functions([module.run.get_concrete_function()], module)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
tflite_bytes = NB12["rename_io"](converter.convert())


def make_interpreter(model_bytes):
    # tensorflow <= 2.16 trên Linux tự gắn Flex delegate; từ 2.20 (mọi OS) và ai_edge_litert thì không.
    return tf.lite.Interpreter(model_content=model_bytes)


_it = make_interpreter(tflite_bytes)
try:                       # op Flex chỉ lỗi lúc invoke, nên chạy thử hẳn một dòng
    _probe = make_interpreter(tflite_bytes)
    _probe.resize_tensor_input(0, [1, len(COLS)])
    _probe.allocate_tensors()
    _probe.set_tensor(_probe.get_input_details()[0]["index"],
                      np.array([[MISSING.encode()] * len(COLS)], dtype=object))
    _probe.invoke()
    FLEX_OK = True
except RuntimeError as exc:
    FLEX_OK = False
    print("Interpreter ở máy này KHÔNG có Flex delegate — mục 5 đối chiếu bằng graph TF eager, "
          "file .tflite kiểm lại trên máy có Flex bằng parity_inputs.npz (mục 6).")
OPS = sorted({op["op_name"] for op in _it._get_ops_details()})
FLEX_OPS = [op for op in OPS if op.startswith("Flex")]
io = (_it.get_input_details()[0], _it.get_output_details()[0])
assert (io[0]["name"], io[1]["name"]) == IO_NAMES
print(f"TFLite {len(tflite_bytes) / 1024:.0f} KiB | input {io[0]['name']} {io[0]['dtype'].__name__} "
      f"{list(io[0]['shape_signature'])} | output {io[1]['name']} {list(io[1]['shape_signature'])}")
print("builtin:", [op for op in OPS if not op.startswith("Flex")])
print("Flex   :", FLEX_OPS)

INFO:tensorflow:Assets written to: C:\Users\admin\AppData\Local\Temp\tmpv0416oig\assets


INFO:tensorflow:Assets written to: C:\Users\admin\AppData\Local\Temp\tmpv0416oig\assets


Interpreter ở máy này KHÔNG có Flex delegate — mục 5 đối chiếu bằng graph TF eager, file .tflite kiểm lại trên máy có Flex bằng parity_inputs.npz (mục 6).
TFLite 3250 KiB | input input bytes_ [np.int32(-1), np.int32(12)] | output output [np.int32(-1), np.int32(3)]
builtin: ['ADD', 'ADD_N', 'ARG_MAX', 'CAST', 'CONCATENATION', 'DIV', 'EQUAL', 'EXPAND_DIMS', 'FILL', 'FULLY_CONNECTED', 'GATHER', 'GREATER', 'GREATER_EQUAL', 'LESS', 'LESS_EQUAL', 'LOGICAL_AND', 'LOGICAL_NOT', 'LOGICAL_OR', 'MAXIMUM', 'MEAN', 'MUL', 'NEG', 'NOT_EQUAL', 'ONE_HOT', 'PACK', 'RANGE', 'REDUCE_ANY', 'REDUCE_MAX', 'REDUCE_MIN', 'RESHAPE', 'SELECT_V2', 'SHAPE', 'SQRT', 'STRIDED_SLICE', 'SUB', 'SUM', 'TILE', 'ZEROS_LIKE']
Flex   : ['FlexRaggedTensorToTensor', 'FlexStaticRegexFullMatch', 'FlexStaticRegexReplace', 'FlexStringJoin', 'FlexStringLower', 'FlexStringSplitV2']


## 5. Kiểm tra

`VERIFY = False` (mặc định) chỉ chạy hai phép kiểm nhanh, không cần ONNX: cascade trên **mọi** tổ hợp đầu vào,
và head OS trên bộ capture so với `os_decide()` của 05b — rồi in mẫu output. `VERIFY = True` chạy thêm đối chiếu
đầy đủ dưới đây (vài phút).

**Chuẩn so** là đường Python đang dùng: nhãn của `sdc_iden.onnx`; khi có `OS_RUN` thì thêm `os_decide()` của
05b (trên khung dựng từ đúng ma trận 12 chuỗi, cờ tự tính như 09) và `cascade()` của 05b. Dữ liệu:

* `sessions_verified` — toàn bộ dữ liệu train, dựng input bằng `add_l0_columns` + `to_input_matrix` của 09;
* bộ pcap tự thu — `dhcp_hostname` thật;
* **dòng ép L0 iden** — hostname khớp từng luật, mDNS `model=`, OUI trong bảng, OUI có bit ngẫu nhiên, xung
  đột mDNS/OUI (bộ dữ liệu thật gần như không chạm L0);
* **dòng ép OS** (khi có `OS_RUN`) — mọi luật `HOST_OS` (kể cả hoa/thường, khoảng trắng, chữ Việt sát biên `\b`),
  VCI `msft`/`android-dhcp`, biến thể PRL quanh từng chữ ký (đủ, đảo thứ tự, thiếu một, thêm 162, trùng, số 0
  đầu), mDNS Apple — mỗi thứ với MAC thật / ngẫu nhiên;
* **fuzz** — mỗi cột rút ngẫu nhiên từ mọi giá trị trên + chuỗi lạ, cộng các dòng dựng thẳng từ khoá vân tay L1.

| Cột | So cái gì | Chạy khi |
|---|---|---|
| `Δx iden` / `Δx OS` | vector feature của graph TF với `ColumnTransformer` | luôn |
| `iden lệch` | nhãn head iden (trước cascade) của graph TF với ONNX | luôn |
| `os lệch` | OS của graph TF với `os_decide()` | có `OS_RUN` |
| `<head> lệch (TF)` | output của graph TF (eager) với chuẩn | luôn |
| `<head> lệch (TFLite)` | output của **file `.tflite`** với chuẩn | interpreter có Flex delegate |

Op Flex trong file chính là kernel TensorFlow, nên cột TF đã kiểm cùng một đoạn code tính toán; cột TFLite
kiểm thêm phần chuyển đổi và runtime. Gói `tensorflow` từ 2.20 (Windows lẫn Linux) **không còn** gắn Flex
delegate vào `tf.lite.Interpreter`; `tensorflow==2.16.2` trên Linux (Python 3.12) thì còn. Không có Flex thì mục
6 ghi `parity_inputs.npz` để chạy cột TFLite ở máy có:

```python
import numpy as np, tensorflow as tf                      # tensorflow==2.16.2 trên Linux: có sẵn Flex
d = np.load("parity_inputs.npz", allow_pickle=True)
it = tf.lite.Interpreter(model_path="sdc_iden_os.tflite")    # hoặc sdc_iden.tflite
for name in d["names"]:
    m, want = d[f"{name}/input"], d[f"{name}/expected"]
    it.resize_tensor_input(0, m.shape); it.allocate_tensors()
    it.set_tensor(it.get_input_details()[0]["index"], np.vectorize(str.encode, otypes=[object])(m))
    it.invoke()
    got = it.get_tensor(it.get_output_details()[0]["index"]).astype(str)
    print(name, (got != want).sum(axis=0))                 # phải là [0 0 0]
```

In [7]:
VERIFY = False   # True = đối chiếu đầy đủ với sdc_iden.onnx + os_decide() trên ~20k dòng (vài phút)

if OS_RUN:
    # Cascade: mọi tổ hợp (OS, MAC ngẫu nhiên, make, type, model) — graph so với cascade() của 05b
    combos = list(itertools.product(range(len(OS_LABELS)), (False, True), range(len(LABELS["make"])),
                                    range(len(LABELS["type"])), range(len(LABELS[MODEL_HEAD]))))
    o, r, mk, ty, mo = (np.asarray(v) for v in zip(*combos))
    got = [t.numpy() for t in module.cascade_fn(tf.constant(o, tf.int32), tf.constant(r), tf.constant(mk, tf.int32),
                                                tf.constant(ty, tf.int32), tf.constant(mo, tf.int32))]
    bad = 0
    for i in range(len(combos)):
        want = NB05B["cascade"](OS_LABELS[o[i]], int(r[i]), LABELS["make"][mk[i]], LABELS["type"][ty[i]],
                                LABELS[MODEL_HEAD][mo[i]])[1:4]
        bad += want != (LABELS["make"][got[0][i]], LABELS["type"][got[1][i]], LABELS[MODEL_HEAD][got[2][i]])
    print(f"cascade: {len(combos)} tổ hợp, lệch {bad}")
    assert bad == 0

cascade: 7560 tổ hợp, lệch 0


In [8]:
def capture_frame():
    feat = DATA / "features"
    sess = pd.read_csv(feat / "session_features_capture.csv")
    dhcp = pd.read_csv(feat / "dhcp_features_capture.csv")
    hostnames = dhcp.dropna(subset=["hostname"]).drop_duplicates("mac").set_index("mac")["hostname"]
    sess["dhcp_hostname"] = sess["mac"].map(hostnames).fillna(MISSING)
    sess["mdns_model"] = MISSING
    return NB09["add_l0_columns"](sess)


def as_matrix(rows):
    return np.asarray([[str(r[c]) for c in COLS] for r in rows], dtype=object)


def iden_probe(base):
    """Dòng ép L0 của iden: mọi luật hostname x (mDNS model=) x (OUI) trên vài dòng nền thật."""
    hostnames = ["Johns-iPhone", "iphone", "DESKTOP-AB12CD3", "desktop-ducanh", "LAPTOP-VAF70SQ6",
                 "Redmi-Note-10", "redmi note 14 pro", "OPPO-A92", "oppo_a5", "raspberrypi", "raspberrypi4-64",
                 "android-0123456789abcdef", "Galaxy-A51", "samsung-cua-duc-anh", "MacBook-Pro", "Thu-iPad",
                 "poco_f3", "xiaomi 13", "linova", MISSING]
    mdns = [MISSING, *list(T["mdns_model"])[:3], "iPhone99,9"]
    ouis = [MISSING, *T["oui"]]
    ouis += [f"{int(o[:2], 16) | 0x02:02x}{o[2:]}" for o in list(T["oui"])[:2]]   # bit U/L bật
    return as_matrix(dict(b, dhcp_hostname=h, mdns_model=m, mac_oui=o)
                     for b in base for h in hostnames for m in mdns for o in ouis)


OS_PROBE = {
    "hostname": ["iPhone", " iPhone ", "IPHONE-cua-Lan", "iphoneđ", "iphone-đ", "ipad-pro", "ipod", "MacBook-Air",
                 "imac", "mac-mini", "macpro", "android-0123456789abcdef", "Galaxy-A51", "sm-a515f", "redmi-9",
                 "xiaomi", "poco-x3", "mi-9", "realme-6i", "oppo-a92", "vivo-1906", "oneplus", "pixel-7",
                 "huawei-p30", "honor-x8", "nokia", "moto-g", "DESKTOP-AB12CD3", "desktop-ducanh", "laptop-vaf70sq6",
                 "win-abcd1234", "WIN-AB", "raspberrypi", "raspberrypi4", "ubuntu", "debian-x", "fedora",
                 "archlinux", "ESP_0C3994", "tuya-plug", "shelly1", "tasmota", "wled-1", "linova", "Johns-iPhone",
                 "ĐIỆN-THOẠI", "none", "NaN", "", MISSING],
    "dhcp_vci": ["MSFT 5.0", " msft 5.0", "MSFT5.0", "android-dhcp-11", "ANDROID-DHCP-16", "android-dhcp",
                 "dhcpcd-6.8.2:Linux-4.4.22+:armv7l", "udhcp 1.26.2", MISSING],
    "dhcp_prl": [],
    "mdns_tokens": ["_companion link _tcp local", "_rdlink _tcp local", "x _companion link",
                    "_airplay _tcp local", MISSING],
}
if OS_RUN:
    for key in os_bundle["signatures"]:
        opts = [str(v) for v in sorted(key)]
        OS_PROBE["dhcp_prl"] += [",".join(opts), ",".join(reversed(opts)), ",".join(opts[:-1]),
                                 ",".join(opts + ["162"]), ",".join([opts[0]] + opts), "0" + ",".join(opts)]


def os_probe(base):
    """Dòng ép từng tầng của head OS, mỗi tầng với MAC thiếu / thật / ngẫu nhiên (OUI thô) / lạ."""
    real = next(iter(T["oui"]))
    ouis = [MISSING, real, f"{int(real[:2], 16) | 0x02:02x}{real[2:]}", "00:11:22", "AA:BB:CC"]
    rows = []
    for b, o in itertools.product(base, ouis):
        rows += [dict(b, dhcp_hostname=h, mac_oui=o) for h in OS_PROBE["hostname"]]
        rows += [dict(b, dhcp_hostname=MISSING, dhcp_vci=v, mac_oui=o) for v in OS_PROBE["dhcp_vci"]]
        rows += [dict(b, dhcp_hostname=MISSING, dhcp_vci=MISSING, dhcp_prl=p, mac_oui=o)
                 for p in OS_PROBE["dhcp_prl"]]
        rows += [dict(b, dhcp_hostname=MISSING, dhcp_vci=MISSING, dhcp_prl=MISSING, mdns_tokens=m, mac_oui=o)
                 for m in OS_PROBE["mdns_tokens"]]
    return as_matrix(rows)


def fuzz_matrix(pools, n, seed=0):
    rng = np.random.default_rng(seed)
    junk = ["", "nan", "None", "x", "1,3,6", "0303|1|h2", "ZZZ unknown token", "1,,3", "01,3"]
    rows = [[str(rng.choice(pools[c] + junk)) for c in COLS] for _ in range(n)]
    for mode, cols, _ in NB09["FP_MODES"]:           # dòng trúng khoá L1, kể cả khoá nhập nhằng
        for head in HEADS:
            for key in T["l1"][mode].get(head, {}):
                r = [str(rng.choice(pools[c])) for c in COLS]
                for c, part in zip(cols, key.split(NB09["KEY_SEP"])):
                    r[COLS.index(c)] = part
                rows.append(r)
    return np.asarray(rows, dtype=object)


capture = capture_frame()
real_frames = {"capture": capture}
MATRICES = {"capture": NB09["to_input_matrix"](capture, COLS)}
if VERIFY:
    sessions = NB09["add_l0_columns"](load_sessions(SESSIONS_PATH)[0])
    real_frames["sessions_verified"] = sessions
    real = pd.concat([sessions[COLS], capture[COLS]], ignore_index=True).astype(str)
    pools = {c: sorted(set(real[c])) for c in COLS}
    pools["mac_oui"] += list(T["oui"])
    pools["mdns_model"] += list(T["mdns_model"])
    base = (sessions.sample(3, random_state=0)[COLS].to_dict("records")
            + capture.sample(2, random_state=0)[COLS].to_dict("records"))
    MATRICES = {"sessions_verified": NB09["to_input_matrix"](sessions, COLS), **MATRICES,
                "ép L0 iden": iden_probe(base)}
    if OS_RUN:
        MATRICES["ép OS"] = os_probe(base)
        for c, values in OS_PROBE.items():
            pools[c] += values
    MATRICES["fuzz"] = fuzz_matrix(pools, 3000)
print({k: m.shape for k, m in MATRICES.items()})

if OS_RUN:
    # Quy ước mac_oui -> mac_is_random phải khớp mac_is_random(MAC) của 05b trên mọi dòng thật
    for name, frame in real_frames.items():
        rule = np.array([mac_random_from_oui(v) for v in frame["mac_oui"]])
        truth = np.array([NB05B["mac_is_random"](m) for m in frame["mac"]])
        print(f"mac_is_random từ mac_oui — {name}: lệch {int((rule != truth).sum())}/{len(frame)} "
              f"(ngẫu nhiên {int(truth.sum())})")
        assert (rule == truth).all()

{'capture': (62, 12)}
mac_is_random từ mac_oui — capture: lệch 0/62 (ngẫu nhiên 25)


In [9]:
BATCH = 256
onnx_sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])


def run_onnx(m):
    return onnx_sess.run([IO_NAMES[1]], {IO_NAMES[0]: m})[0].astype(str)


def reference(m):
    """Chuẩn so: nhãn ONNX, rồi (có OS_RUN) os_decide() + cascade() của 05b."""
    iden = run_onnx(m)
    if not OS_RUN:
        return {"expected": iden, "iden": iden}
    frame = NB09["matrix_frame"](m, COLS, SPEC["iden"]["num"])
    frame["mac_is_random"] = [mac_random_from_oui(v) for v in frame["mac_oui"]]
    decided = NB05B["os_decide"](os_bundle, frame)
    out = [NB05B["cascade"](d[0], rnd, *row)[1:4]
           for d, rnd, row in zip(decided, frame["mac_is_random"], iden)]
    return {"expected": np.asarray(out, dtype=str).reshape(-1, 3), "iden": iden,
            "os": np.asarray([d[0] for d in decided]),
            "x_os": apply_encoder(frame[os_bundle["cols"]], os_bundle["encoder"])[0]}


def run_tflite(m, model_bytes=None):
    it = make_interpreter(model_bytes or tflite_bytes)
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    res = []
    for s in range(0, len(m), BATCH):
        chunk = np.asarray([[v.encode("utf-8") for v in r] for r in m[s:s + BATCH]], dtype=object)
        it.resize_tensor_input(inp["index"], list(chunk.shape))
        it.allocate_tensors()
        it.set_tensor(inp["index"], chunk)
        it.invoke()
        res.append(np.vectorize(lambda b: b.decode("utf-8"))(it.get_tensor(out["index"])))
    return np.concatenate(res)


def run_debug(m):
    parts = [module.debug(tf.constant(m[s:s + BATCH].astype(str))) for s in range(0, len(m), BATCH)]
    return {k: np.concatenate([p[k].numpy() for p in parts]) for k in parts[0]}


def labels_of(idx):
    return np.stack([np.asarray(LABELS[h])[idx[:, i]] for i, h in enumerate(HEADS)], axis=1)


EXPECTED, parity = {}, None
if VERIFY:
    rows = []
    for name, m in MATRICES.items():
        ref, dbg = reference(m), run_debug(m)
        EXPECTED[name] = ref["expected"]
        X_sk = NB09["encode"](NB09["matrix_frame"](m, COLS, SPEC["iden"]["num"]), enc)
        row = {"nguồn": name, "dòng": len(m), "Δx iden": float(np.abs(dbg["x_iden"] - X_sk).max()),
               "iden lệch": int((labels_of(dbg["iden"]) != ref["iden"]).sum())}
        if OS_RUN:
            row["Δx OS"] = float(np.abs(dbg["x_os"] - ref["x_os"]).max())
            row["os lệch"] = int((np.asarray(OS_LABELS)[dbg["os"]] != ref["os"]).sum())
        runs = {"TF": labels_of(dbg["final"])}
        if FLEX_OK:
            runs["TFLite"] = run_tflite(m)
        for kind, got in runs.items():
            for i, h in enumerate(HEADS):
                row[f"{h} lệch ({kind})"] = int((ref["expected"][:, i] != got[:, i]).sum())
        if OS_RUN:
            row["ô đổi bởi cascade"] = int((ref["expected"] != ref["iden"]).sum())
        row["__unknown__"] = int((ref["expected"] == UNK).sum())
        rows.append(row)
    parity = pd.DataFrame(rows).set_index("nguồn")
    display(parity)
    assert parity.filter(regex=r"lệch").to_numpy().sum() == 0, "graph lệch chuẩn"
    print("PARITY OK —", "file .tflite" if FLEX_OK else "graph TF (file .tflite: chạy parity_inputs.npz trên máy có Flex)",
          "trả đúng từng nhãn như chuẩn Python")
else:
    print("VERIFY = False — bỏ qua đối chiếu đầy đủ")

VERIFY = False — bỏ qua đối chiếu đầy đủ


In [10]:
# Mẫu output: các cửa sổ của bộ capture, trước và sau cascade
m = MATRICES["capture"]
dbg = run_debug(m)
sample = pd.DataFrame({"dhcp_hostname": m[:, COLS.index("dhcp_hostname")],
                       "mac ngẫu nhiên": [mac_random_from_oui(v) for v in m[:, COLS.index("mac_oui")]]})
if OS_RUN:
    # kiểm nhanh, không cần ONNX: head OS của graph so với os_decide() của 05b trên chính các dòng này
    frame = NB09["matrix_frame"](m, COLS, SPEC["iden"]["num"])
    frame["mac_is_random"] = [mac_random_from_oui(v) for v in frame["mac_oui"]]
    want_os = np.asarray([d[0] for d in NB05B["os_decide"](os_bundle, frame)])
    got_os = np.asarray(OS_LABELS)[dbg["os"]]
    print(f"head OS trên capture: lệch os_decide() {int((got_os != want_os).sum())}/{len(m)}")
    assert (got_os == want_os).all()
    sample["os"] = got_os
    sample["iden (make/type/model)"] = ["/".join(r) for r in labels_of(dbg["iden"])]
for i, h in enumerate(HEADS):
    sample[h] = labels_of(dbg["final"])[:, i]
display(sample.drop_duplicates().head(20))

head OS trên capture: lệch os_decide() 0/62


,dhcp_hostname,mac ngẫu nhiên,os,iden (make/type/model),make,type,model
0,lee_kingdom,0,__unknown__,Generic Laptop/Laptop/Windows Desktop DELL,Generic Laptop,Laptop,Windows Desktop DELL
1,iphone,1,iOS,Apple/Smartphone/iPhone,Apple,Smartphone,iPhone
2,linova,0,__unknown__,Linova/Linux/Laptop/Linova Laptop HP,Linova/Linux,Laptop,Linova Laptop HP
3,desktop-ducanh,0,Windows,Generic Laptop/Laptop/Windows Desktop DELL,Generic Laptop,Laptop,Windows Desktop DELL
4,<missing>,0,__unknown__,Generic Laptop/Laptop/Windows Laptop HP,Generic Laptop,Laptop,Windows Laptop HP
5,desktop-dngjhrt,0,Windows,Generic Laptop/Laptop/Windows Desktop HP,Generic Laptop,Laptop,Windows Desktop HP
6,<missing>,0,Embedded,Camera/IP Camera/Generic IP Camera,Camera,IP Camera,Generic IP Camera
7,<missing>,1,Android,Xiaomi/Smartphone/Redmi Note 10,Xiaomi,Smartphone,Redmi Note 10
8,samsung-cua-duc-anh,1,Android,Samsung/Smartphone/Samsung Galaxy,Samsung,Smartphone,Samsung Galaxy
9,raspberrypi,0,Linux,Raspberry Pi/Single-board Computer/Raspberry Pi,Raspberry Pi,Single-board Computer,Raspberry Pi


## 6. Nhúng contract và lưu

Contract của ONNX được giữ nguyên các khoá chung (cột, nhãn, hash, hierarchy, ngưỡng) và thêm phần riêng
TFLite: input/output, danh sách op Flex, sha256 của file ONNX nguồn, và (có `OS_RUN`) toàn bộ luật head OS +
cascade + quy ước `mac_is_random`. Ngưỡng **không** ghi đè được lúc chạy như bản ONNX — đổi ngưỡng là chạy lại
notebook này.

**Phía thiết bị:** cần LiteRT build kèm Flex delegate (`tensorflow/lite/delegates/flex`, hoặc AAR
`tensorflow-lite-select-tf-ops` trên Android). Trên Linux/ARM phải tự build `libtensorflowlite_flex.so` —
nặng hơn onnxruntime rất nhiều; với router OpenWrt nên giữ `sdc_iden.onnx`.

In [11]:
contract = {k: v for k, v in onnx_contract.items() if k not in ("size_optimization",)}
contract.update({
    "format": "sdc-iden-os-tflite-v1" if OS_RUN else "sdc-iden-tflite-v1",
    "created": datetime.now().isoformat(timespec="seconds"),
    "source_onnx": {"run_id": iden_dir.name, "file": onnx_path.name,
                    "sha256": hashlib.sha256(onnx_path.read_bytes()).hexdigest()},
    "input": {"name": IO_NAMES[0], "dtype": "string", "shape": [-1, len(COLS)], "columns": COLS},
    "output": {"name": IO_NAMES[1], "dtype": "string", "shape": [-1, len(HEADS)], "columns": list(HEADS)},
    "thresholds_override": False,
})
if OS_RUN:
    contract["os_head"] = {
        "run_id": OS_RUN, "classes": OS_LABELS, "forest_threshold": float(os_bundle["threshold"]),
        "mac_is_random": "từ cột mac_oui: <missing>/rỗng = MAC ngẫu nhiên hoá (router luôn biết MAC, extractor "
                         "chỉ trả <missing> khi MAC ngẫu nhiên); OUI gửi nguyên thì xét bit U/L 0x02 của octet đầu",
        "l0_hostname": [{"search": p, "os": o} for p, o in NB05B["HOST_OS"]],
        "l0_vci": [{"prefix": "msft ", "os": "Windows"},
                   {"prefix": "android-dhcp-", "os": "Android", "require_mac_random": True}],
        "l1_prl": {"min_jaccard": MIN_JACCARD,
                   "signatures": [{"options": sorted(int(v) for v in k), "os": sorted(o)}
                                  for k, o in os_bundle["signatures"].items()]},
        "l1_mdns": {"contains_any": ["_companion link", "_rdlink"], "os": "iOS", "require_mac_random": True},
        "forest_gate": "lớp khác Embedded thì cửa sổ phải có DHCP hoặc mDNS",
        "cascade": {"type_ok": {k: sorted(v) for k, v in NB05B["TYPE_OK"].items()},
                    "general_purpose_types": sorted(NB05B["GENERAL_PURPOSE_TYPES"]),
                    "make_ok": {k: sorted(v) for k, v in NB05B["MAKE_OK"].items()},
                    "fill": {"iOS": "Smartphone + Apple", "Android + MAC ngẫu nhiên": "Smartphone",
                             "Windows": "Laptop", "macOS": "Laptop + Apple"},
                    "model": "make hoặc type là __unknown__ thì model cũng __unknown__"},
    }
contract["runtime"] = {"requires": "LiteRT + Flex delegate (SELECT_TF_OPS)", "flex_ops": FLEX_OPS,
                       "builtin_ops": [op for op in OPS if op not in FLEX_OPS]}
contract_bytes = json.dumps(contract, ensure_ascii=False, indent=1).encode("utf-8")
final_bytes = tflite_bytes
for key, value in ((NB12["CONTRACT_KEY"], contract_bytes),
                   ("sdc.columns_sha256", contract["columns_sha256"].encode()),
                   ("sdc.labels_sha256", contract["labels_sha256"].encode()),
                   ("sdc.format", contract["format"].encode())):
    final_bytes = NB12["embed_metadata"](final_bytes, key, value)

assert NB12["read_metadata"](final_bytes, NB12["CONTRACT_KEY"]) == contract_bytes
_final = make_interpreter(final_bytes)
assert (_final.get_input_details()[0]["name"], _final.get_output_details()[0]["name"]) == IO_NAMES
if FLEX_OK:
    check = MATRICES["capture"]
    assert (run_tflite(check, final_bytes) == run_tflite(check)).all()

out_dir.mkdir(exist_ok=True)
(out_dir / f"{ARTIFACT}.tflite").write_bytes(final_bytes)
(out_dir / f"{ARTIFACT}_contract.json").write_bytes(contract_bytes)
if VERIFY:
    np.savez_compressed(out_dir / "parity_inputs.npz", names=np.array(list(MATRICES)),
                        **{f"{k}/input": v.astype(str) for k, v in MATRICES.items()},
                        **{f"{k}/expected": v for k, v in EXPECTED.items()})
(out_dir / "meta.json").write_text(json.dumps({
    "format": contract["format"], "created": contract["created"], "source_onnx": contract["source_onnx"],
    "os_run": OS_RUN, "tensorflow": tf.__version__, "tflite_bytes": len(final_bytes),
    "contract_sha256": hashlib.sha256(contract_bytes).hexdigest(),
    "ops": OPS, "flex_ops": FLEX_OPS,
    "parity_tflite_checked_here": FLEX_OK,
    "parity": (parity.reset_index().to_dict("records") if parity is not None
               else "không chạy (VERIFY = False): chỉ kiểm cascade mọi tổ hợp + head OS trên capture"),
}, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
for p in sorted(out_dir.iterdir()):
    print(f"{p.stat().st_size / 1024:9.1f} KiB  {p}")

      1.5 KiB  D:\01.AI_Security\02.SDC\Models\20261002_163153_os\tflite_iden_os\meta.json
   3262.9 KiB  D:\01.AI_Security\02.SDC\Models\20261002_163153_os\tflite_iden_os\sdc_iden_os.tflite
     12.9 KiB  D:\01.AI_Security\02.SDC\Models\20261002_163153_os\tflite_iden_os\sdc_iden_os_contract.json
